# Limpeza dos dados
Notebook responsável pela limpeza dos dados em `/data/processed/expired_items_dataset.csv`.

---
# 1. Importando Bibliotecas

Bibliotecas utilizadas para manipulação dos dados e controle de avisos.

In [1]:
# importando bibliotecas 

import pandas as pd
import warnings 
import unicodedata

# ignorando avisos

warnings.filterwarnings(
    "ignore", category=FutureWarning
    )

---
# 2. Explorando os Dados

Exploração inicial do conjunto de dados `expired_items_dataset.csv`

In [2]:
# importanto informações do dataset

df = pd.read_csv(
    '../data/processed/expired_items_dataset.csv'
)

# verificando estrutura do dataframe

df.shape

(28006, 7)

Carregamento do DataSet.

O conjunto de dados contém 28.006 registros e 9 colunas e será utilizado para a análise exploratória inicial.

In [3]:
# visualização superficial do DataFrame 

df.head(10)

,DATA,BANDEIRA,LOJA,QTDE.,SETOR,VALIDADE,DIAS VENCIDO
0,3/1/2022,JOAO,16,1,MERCEARIA COMPLEMENTAR,20/10/21,-75
1,3/1/2022,JOAO,16,4,MERCEARIA COMPLEMENTAR,1/1/2022,-2
2,3/1/2022,JOAO,16,1,MERCEARIA COMPLEMENTAR,1/12/2021,-33
3,3/1/2022,JOAO,16,2,MERCEARIA COMPLEMENTAR,1/1/2022,-2
4,3/1/2022,JOAO,16,3,MERCEARIA COMPLEMENTAR,30/04/21,-248
5,3/1/2022,JOAO,16,2,MERCEARIA COMPLEMENTAR,8/7/2021,-179
6,3/1/2022,JOAO,16,13,MERCEARIA COMPLEMENTAR,13/11/21,-51
7,3/1/2022,JOAO,16,1,MERCEARIA COMPLEMENTAR,12/5/2021,-236
8,3/1/2022,JOAO,16,1,MERCEARIA COMPLEMENTAR,31/12/21,-3
9,3/1/2022,JOAO,16,1,MERCEARIA COMPLEMENTAR,16/12/21,-18


Visualização das 10 primeiras linhas do DataFrame. 

Principais ajustes identificados:
- Padronização de colunas de data
- Tratamento de colunas de texto
- Remoção de colunas vazias

In [4]:
# retorna o nome das colunas

df.columns

Index(['DATA', 'BANDEIRA', 'LOJA', 'QTDE.', 'SETOR', 'VALIDADE',
       'DIAS VENCIDO'],
      dtype='object')

Listagem do nome das colunas para referência durante renomeação.

In [5]:
# retorna informações gerais do dataframe

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28006 entries, 0 to 28005
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   DATA          28006 non-null  object
 1   BANDEIRA      28006 non-null  object
 2   LOJA          28006 non-null  int64 
 3   QTDE.         28006 non-null  int64 
 4   SETOR         28006 non-null  object
 5   VALIDADE      28006 non-null  object
 6   DIAS VENCIDO  28006 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 1.5+ MB


Inspeção da estrutura da base para identificar tipos de dados presentes e valores nulos.

In [6]:
# quantidade de valores unicos de cada col 

df.nunique()

DATA             919
BANDEIRA           2
LOJA              23
QTDE.            212
SETOR             33
VALIDADE        1589
DIAS VENCIDO     405
dtype: int64

In [7]:
# valores unicos das colunas
print(
    'BANDEIRA:', df['BANDEIRA'].unique()
)

print(
    'LOJA:', df['LOJA'].unique()
)

print(
    'SETOR:', df['SETOR'].unique()
)

BANDEIRA: ['JOAO' 'SEBASTIAO']
LOJA: [16 34  7 40 17  9 12 14 25 30 21  8 15  5 23  4 42 36 31 32 22 35 20]
SETOR: ['MERCEARIA COMPLEMENTAR' 'PERECÍVEIS COMPLEMENTARES' 'PERECÍVEIS FLV'
 'PERECÍVEIS LATICÍNIOS' 'PERECÍVEIS PADARIA E CONFEITARIA'
 'PERECÍVEIS CONGELADOS' 'PERECÍVEIS AÇOUGUE' 'PERECÍVEIS ROTISSERIA'
 'PERECÍVEIS PEIXARIA' 'MERCEARIA LÍQUIDA' 'MERCEARIA LIMPEZA'
 'BAZAR CASA' 'MERCEARIA PERFUMARIA' 'MERCEARIA BÁSICA'
 'FOOD SERVICE MERCEARIA' 'BAZAR JARDINAGEM' 'PERECÍVEIS FOOD SERVICE'
 'mercearia complementar' 'food service mercearia' 'bazar jardinagem'
 'perecíveis flv' 'MERCEARIA LíQUIDA' 'MERCEARIA LIMPEZA '
 'MERCEARIA PERFUMARIA ' 'MERCEARIA BÁSICA ' 'PERECÍVEIS LATICÍNIOS '
 'PERECÍVEIS AÇOUGUE ' 'PERECÍVEIS ROTISSERIA '
 'PERECÍVEIS COMPLEMENTARES ' 'MERCEARIA COMPLEMENTAR '
 'MERCEARIA FOOD SERVICE' 'MERCEARI COMPLEMENTAR ' 'MERCEARIA LÍQUIDA ']


Após analisar os valores únicos principais do DataFrame, é reforçada a necessidade de tratamento das colunas de texto, em especial a coluna `SETORES` que é de extrema relavância para análises posteriores.

In [8]:
# conferindo se o calculo do dataset foi feito corretamente

df['DIAS VENCIDO'].sort_values(ascending=False).head(10)

6070     356
15900    144
19376    114
4609      82
15977     64
4126      57
15902     46
15785     45
15752     45
15753     32
Name: DIAS VENCIDO, dtype: int64

Concluímos que o calculo está errado para algumas celulas, o mais seguro é remover a coluna e realizar o calculo no SQL.

 

---
# 3. Limpando os Dados

In [9]:
# removendo colunas 

df.drop(columns=[
    'DIAS VENCIDO'
], inplace=True)

# renomeando colunas

df.rename( columns={
    'QTDE.':'QUANTIDADE'
}, inplace=True)

# confirmando alterações

df.columns

Index(['DATA', 'BANDEIRA', 'LOJA', 'QUANTIDADE', 'SETOR', 'VALIDADE'], dtype='object')

Remoção da coluna com erros e renomeação de `QTDE.` para melhor compreensão.

In [17]:
# define colunas de data 

colunas_data = [
    'DATA',
    'VALIDADE'
]

# conversão para datetime 

df[colunas_data] = df[colunas_data].apply(
    pd.to_datetime,
    dayfirst=True,
    errors='coerce'
)

# confirmando conversão

print(df[colunas_data].dtypes)
print(df[colunas_data].head(4))

DATA        datetime64[ns]
VALIDADE    datetime64[ns]
dtype: object
        DATA   VALIDADE
0 2022-01-03 2021-10-20
1 2022-01-03 2022-01-01
2 2022-01-03 2021-12-01
3 2022-01-03 2022-01-01


Essa conversão não é definitiva, apenas experimental para confirmar se as colunas aceitam conversão sem erros. 

In [11]:
# define colunas str

colunas_texto = [
    'BANDEIRA',
    'SETOR'
]

# padronizando colunas str

#basica
df[colunas_texto] = (
    df[colunas_texto].apply(
        lambda col: col.str.strip().str.upper()        # remove espaços e deixa em caixa alta
    )
)
# remoção de acentos
df[colunas_texto] = (
    df[colunas_texto]
    .apply(lambda col: col.apply(
        lambda x: unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode('utf-8')
    ))
)

# confirmando padronização

df[colunas_texto].head()

,BANDEIRA,SETOR
0,JOAO,MERCEARIA COMPLEMENTAR
1,JOAO,MERCEARIA COMPLEMENTAR
2,JOAO,MERCEARIA COMPLEMENTAR
3,JOAO,MERCEARIA COMPLEMENTAR
4,JOAO,MERCEARIA COMPLEMENTAR


Padronização de texto presente nas colunas, removendo espaços, acentos e alterando para caixa alta.

In [12]:
# confirmando quantidade de valores unicos 

df['SETOR'].nunique()

19

Os valores únicos de `SETOR` cairam de 33 para 19 apenas com padronização de texto. 

In [13]:
# confirmando valores unicos 

df['SETOR'].unique()

array(['MERCEARIA COMPLEMENTAR', 'PERECIVEIS COMPLEMENTARES',
       'PERECIVEIS FLV', 'PERECIVEIS LATICINIOS',
       'PERECIVEIS PADARIA E CONFEITARIA', 'PERECIVEIS CONGELADOS',
       'PERECIVEIS ACOUGUE', 'PERECIVEIS ROTISSERIA',
       'PERECIVEIS PEIXARIA', 'MERCEARIA LIQUIDA', 'MERCEARIA LIMPEZA',
       'BAZAR CASA', 'MERCEARIA PERFUMARIA', 'MERCEARIA BASICA',
       'FOOD SERVICE MERCEARIA', 'BAZAR JARDINAGEM',
       'PERECIVEIS FOOD SERVICE', 'MERCEARIA FOOD SERVICE',
       'MERCEARI COMPLEMENTAR'], dtype=object)

Analisando os valores únicos remanescentes na coluna `SETOR`, ainda é necessário uma alteração.

In [14]:
df.SETOR.replace({
    'MERCEARI COMPLEMENTAR' : 'MERCEARIA COMPLEMENTAR'
}, inplace=True
)

# confirmando alteração

df['SETOR'].unique()

array(['MERCEARIA COMPLEMENTAR', 'PERECIVEIS COMPLEMENTARES',
       'PERECIVEIS FLV', 'PERECIVEIS LATICINIOS',
       'PERECIVEIS PADARIA E CONFEITARIA', 'PERECIVEIS CONGELADOS',
       'PERECIVEIS ACOUGUE', 'PERECIVEIS ROTISSERIA',
       'PERECIVEIS PEIXARIA', 'MERCEARIA LIQUIDA', 'MERCEARIA LIMPEZA',
       'BAZAR CASA', 'MERCEARIA PERFUMARIA', 'MERCEARIA BASICA',
       'FOOD SERVICE MERCEARIA', 'BAZAR JARDINAGEM',
       'PERECIVEIS FOOD SERVICE', 'MERCEARIA FOOD SERVICE'], dtype=object)

In [15]:
df.head()

,DATA,BANDEIRA,LOJA,QUANTIDADE,SETOR,VALIDADE
0,2022-01-03,JOAO,16,1,MERCEARIA COMPLEMENTAR,2021-10-20
1,2022-01-03,JOAO,16,4,MERCEARIA COMPLEMENTAR,2022-01-01
2,2022-01-03,JOAO,16,1,MERCEARIA COMPLEMENTAR,2021-12-01
3,2022-01-03,JOAO,16,2,MERCEARIA COMPLEMENTAR,2022-01-01
4,2022-01-03,JOAO,16,3,MERCEARIA COMPLEMENTAR,2021-04-30


---
# 4. Criação do Novo Arquivo

In [16]:
# salvando o dataframe tratado em novo arquivo csv 

df.to_csv('../data/processed/expired_items_dataset_processed.csv', index=False)

Após todo tratamento, o arquivo tratado é criado e inserido na pasta /data/processed.